# Module 1: Data Ingestion and Exploratory Data Analysis (ISOT Dataset)

**Goal:** Inspect the raw dataset, verify dimensions, check for missing values, analyze class balance, and document initial source and metadata shortcuts.

### Key Caveats Audited in this Module:
- **Caveat 1 (Source Leakage):** Real news articles overwhelmingly start with agency tags like `CITY (Reuters) -`.
- **Caveat 2 (Metadata Leakage):** The `subject` column does not overlap between Real and Fake articles at all, providing a trivial shortcut if not removed.
- **Caveat 3 (Duplicates):** Duplicate records in raw data must be identified and cataloged before splitting.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import RAW_DATA_DIR, FIGURES_DIR, SEED, set_seed
from src.preprocess import load_raw_data, get_data_summary
from src.run_eda import generate_eda_figures

set_seed(SEED)
print(f"Project root configured: {project_root}")

## 1. Load Raw Datasets
We load `True.csv` (label=1) and `Fake.csv` (label=0) and concatenate them into a single dataframe.

In [ ]:
df = load_raw_data(RAW_DATA_DIR)
summary = get_data_summary(df)
print(f"Total articles: {summary['total_rows']:,}")
print(f"Columns: {summary['columns']}")
print("\nNull counts per column:")
for col, count in summary['null_counts'].items():
    print(f"  {col}: {count}")

print("\nClass counts:")
for label, count in summary['class_counts'].items():
    lbl_name = 'Real' if label == 1 else 'Fake'
    pct = summary['class_proportions'][label] * 100
    print(f"  {lbl_name} ({label}): {count:,} ({pct:.2f}%)")

## 2. Subject Column Shortcut Audit (Caveat 2)
We examine the distribution of `subject` across classes. Notice that there is ZERO cross-class overlap.

In [ ]:
subject_cross = pd.crosstab(df['subject'], df['label']).rename(columns={1: 'Real', 0: 'Fake'})
print("Subject crosstab (Real vs Fake):")
display(subject_cross)
print("\nCRITICAL FINDING: Real articles belong ONLY to 'politicsNews' and 'worldnews'.")
print("Fake articles belong ONLY to 'Government News', 'Middle-east', 'News', 'US_News', 'left-news', and 'politics'.")
print("Conclusion: Keeping 'subject' would cause 100% artificial test score leakage. It MUST be dropped in Module 2.")

## 3. Agency / Wire Source Tag Audit (Caveat 1)
Real news articles almost universally begin with agency datelines like `WASHINGTON (Reuters) -`.

In [ ]:
reuters_pattern = r"reuters"
df['has_reuters'] = df['text'].fillna("").str.contains(reuters_pattern, case=False, regex=True)
reuters_rates = df.groupby('label')['has_reuters'].mean() * 100
print(f"Percentage of Real articles mentioning Reuters: {reuters_rates.get(1, 0):.2f}%")
print(f"Percentage of Fake articles mentioning Reuters: {reuters_rates.get(0, 0):.2f}%")
print("\nFirst 3 Real article headers:")
for t in df[df['label'] == 1]['text'].head(3):
    print("  ->", repr(t[:80]))
print("\nFirst 3 Fake article headers:")
for t in df[df['label'] == 0]['text'].head(3):
    print("  ->", repr(t[:80]))

## 4. Text Length and Distribution Analysis
Examining word counts and identifying completely empty texts.

In [ ]:
df['word_count'] = df['text'].fillna("").apply(lambda t: len(str(t).split()))
print("Word count statistics by class:")
display(df.groupby('label')['word_count'].describe())
empty_count = (df['text'].fillna("").str.strip() == "").sum()
print(f"\nNumber of articles with empty or whitespace text: {empty_count}")

## 5. Visualizations & Figures
Generating publication-ready figures for Class Balance, Subject Leakage, Word Count Distribution, and Reuters Fingerprint Rate.

In [ ]:
generate_eda_figures(df, FIGURES_DIR)
print(f"Figures generated and saved to: {FIGURES_DIR}")